## Importing Necessary Libraries

In [359]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
from gensim.models import Word2Vec
from transformers import BertTokenizer, BertModel
import torch

In [360]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Loading Data and Investing

In [361]:
posts_path =r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts.csv"
df = pd.read_csv(r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts.csv")
# Drop column type and image
df = df.drop(columns=['type', 'image'])

## Data Pre-Processing

In [362]:
def overall(df):
    print ("Rows : " ,df.shape[0])
    print ("Columns : " ,df.shape[1])
    print ("\nFeatures : \n" ,df.columns.tolist())
    print ("\nMissing values : ", df.isnull().sum().values.sum())
    print ("\nUnique values : \n", df.nunique())
    
overall(df)

Rows :  827
Columns :  7

Features : 
 ['post_id', 'timestamp', 'ownerUsername', 'caption', 'hashtags', 'likesCount', 'commentsCount']

Missing values :  360

Unique values : 
 post_id          827
timestamp        827
ownerUsername     48
caption          777
hashtags         359
likesCount       787
commentsCount    358
dtype: int64


#### Check missing value (Post Dataframe)

In [363]:
df.isnull().sum()

post_id            0
timestamp          0
ownerUsername      0
caption            7
hashtags         353
likesCount         0
commentsCount      0
dtype: int64

In [364]:
# Fill missing value in caption and hashtags column by ""
df['caption'] = df['caption'].fillna('')
df['hashtags'] = df['hashtags'].fillna('')

In [365]:
df.isnull().sum()

post_id          0
timestamp        0
ownerUsername    0
caption          0
hashtags         0
likesCount       0
commentsCount    0
dtype: int64

In [366]:
df.head()

,post_id,timestamp,ownerUsername,caption,hashtags,likesCount,commentsCount
0,1,2023-06-02 16:34:43+00:00,maryleest,Cannes 2023 with @kilianparis 🤍 #KilianCannes ...,KilianCannes,32070,142
1,2,2023-01-01 07:49:16+00:00,tinaabeysekara,"As the clock struck midnight to ring in 2022, ...",,6565,86
2,3,2023-05-29 18:57:51+00:00,maryleest,The famous stairs 🤎 Photo @gustave_durin Dre...,,28936,123
3,4,2023-03-07 19:30:42+00:00,stephaniebroek,I visualized this moment so many times before....,CHANELFallWinter,4764,215
4,5,2023-05-23 21:11:12+00:00,maryleest,Got to witness such historical moment in cinem...,CannesFilmFestival,13379,123


In [367]:
# Type of each column
df.dtypes

post_id           int64
timestamp        object
ownerUsername    object
caption          object
hashtags         object
likesCount        int64
commentsCount     int64
dtype: object

### "Caption" Column

In [368]:
# Replace emoji by text
#df['caption'] = df['caption'].apply(lambda x: emoji.demojize(x) if isinstance(x, str) else x)
# Lowercase text
#df['caption'] = df['caption'].str.lower()
#punctuation_to_remove = ''.join(ch for ch in string.punctuation if ch not in [':', '_'])
#df['caption'] = df['caption'].apply(lambda x: re.sub(rf"[{re.escape(punctuation_to_remove)}]", '', x))
#df['tokens'] = df['caption'].apply(lambda x: word_tokenize(x))
#stop_words = set(stopwords.words('english'))
#df['tokens'] = df['tokens'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

In [369]:
# Clean caption
def clean_caption(text):
    if pd.isna(text) or text == 'NaN':
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^\w\s\U0001F000-\U0001F9FF]', ' ', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['clean_caption'] = df['caption'].apply(clean_caption)

In [370]:
# Process Emoji in caption
def process_emoji(text):
    emojis_list = [c for c in text if c in emoji.EMOJI_DATA]
    
    emoji_descriptions = []
    for e in emojis_list:
        emoji_name = emoji.demojize(e).replace(':', '').replace('_', ' ')
        emoji_descriptions.append(emoji_name)
    
    text_without_emoji = ''.join(c for c in text if c not in emoji.EMOJI_DATA)
    
    return text_without_emoji, emoji_descriptions

df[['caption_no_emoji', 'emoji_descriptions']] = df['clean_caption'].apply(lambda x: pd.Series(process_emoji(x)))


#### "Hashtag" column

In [371]:
# Process hashtags from hashtag column
def process_hashtags_from_column(hashtag_text):
    if pd.isna(hashtag_text) or hashtag_text == 'nan' or hashtag_text == '':
        return []
    
    if hashtag_text.startswith('[') and hashtag_text.endswith(']'):
        hashtag_text = hashtag_text[1:-1]
        
        tags = [tag.strip().strip("'").strip('"') for tag in hashtag_text.split(',')]
    else:
        tags = [tag.strip().strip('#') for tag in re.split(r'[,#]', hashtag_text) if tag.strip()]
    
    processed_hashtags = []
    for tag in tags:
        if tag:
            words = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', tag)
            if not words:
                words = [tag]
            processed_hashtags.extend([word.lower() for word in words])
    
    return processed_hashtags

df['hashtags'] = df['hashtags'].apply(process_hashtags_from_column)

In [372]:
# Tokenize and remove stopwords
def tokenize_and_remove_stopwords(text):
    
    tokens = word_tokenize(text)
    
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    
    return filtered_tokens

df['caption_tokens'] = df['caption_no_emoji'].apply(tokenize_and_remove_stopwords)

In [373]:
df['combined_tokens'] = df.apply(lambda row: row['caption_tokens'] + row['emoji_descriptions'] + row['hashtags'], axis=1)

In [374]:
df.head()

,post_id,timestamp,ownerUsername,caption,hashtags,likesCount,commentsCount,clean_caption,caption_no_emoji,emoji_descriptions,caption_tokens,combined_tokens
0,1,2023-06-02 16:34:43+00:00,maryleest,Cannes 2023 with @kilianparis 🤍 #KilianCannes ...,"[kilian, cannes]",32070,142,cannes 2023 with kilianparis 🤍 kiliancannes we...,cannes 2023 with kilianparis kiliancannes wea...,[white heart],"[cannes, 2023, kilianparis, kiliancannes, wear...","[cannes, 2023, kilianparis, kiliancannes, wear..."
1,2,2023-01-01 07:49:16+00:00,tinaabeysekara,"As the clock struck midnight to ring in 2022, ...",[],6565,86,as the clock struck midnight to ring in 2022 i...,as the clock struck midnight to ring in 2022 i...,"[woman dancing, medium skin tone, dizzy, mediu...","[clock, struck, midnight, ring, 2022, cried, w...","[clock, struck, midnight, ring, 2022, cried, w..."
2,3,2023-05-29 18:57:51+00:00,maryleest,The famous stairs 🤎 Photo @gustave_durin Dre...,[],28936,123,the famous stairs 🤎 photo gustave_durin dress ...,the famous stairs photo gustave_durin dress m...,[brown heart],"[famous, stairs, photo, gustave_durin, dress, ...","[famous, stairs, photo, gustave_durin, dress, ..."
3,4,2023-03-07 19:30:42+00:00,stephaniebroek,I visualized this moment so many times before....,"[chanel, fall, winter]",4764,215,i visualized this moment so many times before ...,i visualized this moment so many times before ...,"[white heart, black heart]","[visualized, moment, many, times, first, chane...","[visualized, moment, many, times, first, chane..."
4,5,2023-05-23 21:11:12+00:00,maryleest,Got to witness such historical moment in cinem...,"[cannes, film, festival]",13379,123,got to witness such historical moment in cinem...,got to witness such historical moment in cinem...,[broken heart],"[got, witness, historical, moment, cinema, kil...","[got, witness, historical, moment, cinema, kil..."


In [375]:
df_processed = df[['post_id', 'hashtags', 'clean_caption', 'caption_no_emoji','emoji_descriptions', 'caption_tokens', 'combined_tokens']]
df_processed.head(5)

,post_id,hashtags,clean_caption,caption_no_emoji,emoji_descriptions,caption_tokens,combined_tokens
0,1,"[kilian, cannes]",cannes 2023 with kilianparis 🤍 kiliancannes we...,cannes 2023 with kilianparis kiliancannes wea...,[white heart],"[cannes, 2023, kilianparis, kiliancannes, wear...","[cannes, 2023, kilianparis, kiliancannes, wear..."
1,2,[],as the clock struck midnight to ring in 2022 i...,as the clock struck midnight to ring in 2022 i...,"[woman dancing, medium skin tone, dizzy, mediu...","[clock, struck, midnight, ring, 2022, cried, w...","[clock, struck, midnight, ring, 2022, cried, w..."
2,3,[],the famous stairs 🤎 photo gustave_durin dress ...,the famous stairs photo gustave_durin dress m...,[brown heart],"[famous, stairs, photo, gustave_durin, dress, ...","[famous, stairs, photo, gustave_durin, dress, ..."
3,4,"[chanel, fall, winter]",i visualized this moment so many times before ...,i visualized this moment so many times before ...,"[white heart, black heart]","[visualized, moment, many, times, first, chane...","[visualized, moment, many, times, first, chane..."
4,5,"[cannes, film, festival]",got to witness such historical moment in cinem...,got to witness such historical moment in cinem...,[broken heart],"[got, witness, historical, moment, cinema, kil...","[got, witness, historical, moment, cinema, kil..."


In [376]:
from collections import Counter
all_tokens = sum(df['combined_tokens'], []) 
word_freq = Counter(all_tokens)  # Đếm số lần xuất hiện của từng từ

keywords = [word for word, freq in word_freq.items() if freq > 1]
print("Keywords:", keywords)

Keywords: ['cannes', '2023', 'wearing', 'dress', 'white heart', 'clock', 'midnight', '2022', 'year', 'ever', 'kalu', 'fur', '18', 'years', 'given', 'weeks', 'live', 'could', 'yet', 'somehow', 'came', 'pretty', 'close', 'dream', 'beliefs', 'growing', 'jewellery', 'brand', 'saying', 'yes', 'lives', 'moving', 'across', 'world', 'thank', 'supporting', 'welcome', 'new', 'home', 'san', 'francisco', 'emotionally', 'literally', 'person', 'intention', 'play', 'adventure', 'leading', 'works', 'mindset', 'excited', 'share', 'ride', 'following', 'along', 'happy', 'medium skin tone', 'dizzy', 'medium-dark skin tone', 'smiling face with hearts', 'partying face', 'famous', 'photo', 'gustave_durin', 'marmarhalim', 'jewelry', 'yessayan', 'hair', 'balmainhaircouture', 'makeup', 'narsissist', 'brown heart', 'visualized', 'moment', 'many', 'times', 'first', 'chanelofficial', 'show', 'always', 'thought', 'would', 'happen', 'get', 'editor', 'fashion', 'director', 'chief', 'one', 'day', 'role', 'go', 'attend

### BERT Embedding for combined_tokens

In [377]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

c:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\venv\lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [378]:
def get_bert_embedding(tokens):
    """Convert combined_tokens (list of words) to a BERT embedding."""
    text = " ".join(tokens)  # Convert list to a single sentence
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=50)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()  # Take mean embedding

# Apply to dataframe
df_processed['bert_embedding'] = df_processed['combined_tokens'].apply(get_bert_embedding)

print(df_processed[['combined_tokens', 'bert_embedding']].head())

                                     combined_tokens  \
0  [cannes, 2023, kilianparis, kiliancannes, wear...   
1  [clock, struck, midnight, ring, 2022, cried, w...   
2  [famous, stairs, photo, gustave_durin, dress, ...   
3  [visualized, moment, many, times, first, chane...   
4  [got, witness, historical, moment, cinema, kil...   

                                      bert_embedding  
0  [-0.05747311, -0.37669456, 0.27712598, 0.08630...  
1  [0.15091074, -0.07945571, 0.80125374, -0.22111...  
2  [0.04359917, 0.13778722, 0.13084672, -0.280214...  
3  [-0.12334562, -0.13858783, 0.6960118, -0.12609...  
4  [0.2409971, 0.20214732, 0.16738108, 0.00604014...  


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28136\3521289661.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_processed['bert_embedding'] = df_processed['combined_tokens'].apply(get_bert_embedding)


### Handle OOV Words with Word2Vec

In [379]:
from gensim.models import Word2Vec

# Train Word2Vec on combined_tokens
w2v_model = Word2Vec(df['combined_tokens'], vector_size=100, window=5, min_count=1, workers=4)

def get_w2v_embedding(tokens):
    """Average Word2Vec vectors for a sentence."""
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)  # Handle OOV cases

df_processed['word2vec_embedding'] = df_processed['combined_tokens'].apply(get_w2v_embedding)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28136\1223182056.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_processed['word2vec_embedding'] = df_processed['combined_tokens'].apply(get_w2v_embedding)


#### Combine BERT & Word2Vec:

In [380]:
def combine_embeddings(bert_vec, w2v_vec):
    """Combine BERT and Word2Vec embeddings, using Word2Vec when BERT fails."""
    if np.all(bert_vec == 0):  # Nếu BERT không có vector (OOV case)
        return w2v_vec
    return np.concatenate([bert_vec, w2v_vec])  # Kết hợp cả hai

df_processed['final_embedding'] = df_processed.apply(lambda row: combine_embeddings(row['bert_embedding'], row['word2vec_embedding']), axis=1)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28136\1641342415.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_processed['final_embedding'] = df_processed.apply(lambda row: combine_embeddings(row['bert_embedding'], row['word2vec_embedding']), axis=1)


### Apply Attention for Fashion Keywords

In [381]:
fashion_keywords = {
    # 👕 Clothing Items
    "clothing_items": [
        "sweater", "blouse", "jeans", "t-shirt", "shirt", "dress", "skirt", "trousers", 
        "shorts", "cardigan", "hoodie", "jumpsuit", "romper", "suit", "blazer", "coat",
        "jacket", "overcoat", "windbreaker", "leggings", "crop top", "tank top", "bodysuit"
    ],
    
    # 🧵 Materials
    "materials": [
        "cotton", "linen", "denim", "silk", "wool", "polyester", "rayon", "spandex",
        "leather", "suede", "cashmere", "velvet", "satin", "chiffon", "mesh", "lace",
        "knit", "corduroy", "tweed", "nylon", "faux fur", "hemp", "modal", "terrycloth"
    ],
    
    # 🎨 Colors
    "colors": [
        "black", "white", "navy", "beige", "cream", "khaki", "gray", "charcoal", "brown",
        "olive", "mustard", "gold", "silver", "bronze", "teal", "turquoise", "aqua",
        "red", "maroon", "burgundy", "pink", "rose", "peach", "coral", "orange",
        "yellow", "lime", "green", "mint", "forest green", "blue", "royal blue",
        "indigo", "purple", "lavender", "violet", "lilac"
    ],
    
    # 🏷️ Styles
    "styles": [
        "vintage", "bohemian", "minimalist", "streetwear", "casual", "business casual",
        "formal", "elegant", "chic", "grunge", "punk", "gothic", "preppy", "hipster",
        "y2k", "athleisure", "skater", "retro", "avant-garde", "monochrome", "artsy",
        "country", "military", "workwear", "sporty", "kawaii", "coquette"
    ],
    
    # 🛍️ Brands (Luxury & High-Street)
    "brands": [
        "Gucci", "Zara", "Uniqlo", "Prada", "Louis Vuitton", "Chanel", "Hermès", "Dior",
        "Balenciaga", "Versace", "Burberry", "Givenchy", "Dolce & Gabbana", "Fendi",
        "Saint Laurent", "Alexander McQueen", "Tom Ford", "Celine", "Valentino",
        "Armani", "Ralph Lauren", "Lacoste", "Hugo Boss", "Calvin Klein", "Levi's",
        "Nike", "Adidas", "Puma", "New Balance", "Under Armour", "Reebok", "The North Face",
        "Patagonia", "H&M", "Mango", "ASOS", "Bershka", "Forever 21", "Urban Outfitters",
        "Shein", "Victoria’s Secret"
    ],

    # 🖼️ Patterns
    "patterns": [
        "striped", "polka dot", "floral", "paisley", "plaid", "herringbone", "tie-dye",
        "leopard print", "zebra print", "camouflage", "geometric", "tartan", "abstract",
        "checkered", "ombre", "holographic", "baroque"
    ],

    # 👜 Accessories
    "accessories": [
        "hat", "cap", "beanie", "scarf", "belt", "gloves", "earrings", "necklace",
        "bracelet", "rings", "watch", "handbag", "tote bag", "clutch", "backpack",
        "sunglasses", "eyeglasses", "hairband", "headband", "shawl"
    ],

    # 👟 Footwear
    "footwear": [
        "sneakers", "heels", "boots", "sandals", "loafers", "oxfords", "moccasins",
        "flip-flops", "espadrilles", "wedges", "slippers", "platform shoes", "ballet flats"
    ]
}


In [ ]:
import torch.nn as nn

class AttentionLayer(nn.Module):
    """Simple Attention Mechanism"""
    def __init__(self, input_dim):
        super(AttentionLayer, self).__init__()
        self.attention = nn.Linear(input_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attention(x), dim=1)  # Compute attention weights
        return (weights * x).sum(dim=1)  # Weighted sum

# Tính trọng số Attention dựa trên fashion keywords
def get_attention_weights(tokens):
    return [1.5 if token in fashion_keywords else 1.0 for token in tokens]

# Apply attention
df_processed['attention_weights'] = df_processed['combined_tokens'].apply(get_attention_weights)
attention_layer = AttentionLayer(len(df_processed['final_embedding'][0]))  # Kích thước embedding

# Convert embeddings to tensor
embeddings_tensor = torch.tensor(df_processed['final_embedding'].tolist(), dtype=torch.float32)
attention_output = attention_layer(embeddings_tensor)

df_processed['final_fashion_embedding'] = attention_output.tolist()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28136\4274515284.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_processed['attention_weights'] = df_processed['combined_tokens'].apply(get_attention_weights)


In [392]:
df_processed.head()

,post_id,hashtags,clean_caption,caption_no_emoji,emoji_descriptions,caption_tokens,combined_tokens,bert_embedding,word2vec_embedding,final_embedding,attention_weights,final_fashion_embedding
0,1,"[kilian, cannes]",cannes 2023 with kilianparis 🤍 kiliancannes we...,cannes 2023 with kilianparis kiliancannes wea...,[white heart],"[cannes, 2023, kilianparis, kiliancannes, wear...","[cannes, 2023, kilianparis, kiliancannes, wear...","[-0.05747311, -0.37669456, 0.27712598, 0.08630...","[-0.004949781, 0.007430109, 0.0007432284, 0.00...","[-0.05747311, -0.37669456, 0.27712598, 0.08630...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",-7.308448
1,2,[],as the clock struck midnight to ring in 2022 i...,as the clock struck midnight to ring in 2022 i...,"[woman dancing, medium skin tone, dizzy, mediu...","[clock, struck, midnight, ring, 2022, cried, w...","[clock, struck, midnight, ring, 2022, cried, w...","[0.15091074, -0.07945571, 0.80125374, -0.22111...","[-0.0017464517, 0.003086213, -5.9092497e-05, -...","[0.15091074, -0.07945571, 0.80125374, -0.22111...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",-8.383573
2,3,[],the famous stairs 🤎 photo gustave_durin dress ...,the famous stairs photo gustave_durin dress m...,[brown heart],"[famous, stairs, photo, gustave_durin, dress, ...","[famous, stairs, photo, gustave_durin, dress, ...","[0.04359917, 0.13778722, 0.13084672, -0.280214...","[-0.00033619947, 0.0038199516, -0.0025423593, ...","[0.04359917, 0.13778722, 0.13084672, -0.280214...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",-7.752239
3,4,"[chanel, fall, winter]",i visualized this moment so many times before ...,i visualized this moment so many times before ...,"[white heart, black heart]","[visualized, moment, many, times, first, chane...","[visualized, moment, many, times, first, chane...","[-0.12334562, -0.13858783, 0.6960118, -0.12609...","[-0.0033083402, 0.0042623016, -9.507986e-05, -...","[-0.12334562, -0.13858783, 0.6960118, -0.12609...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",-7.661572
4,5,"[cannes, film, festival]",got to witness such historical moment in cinem...,got to witness such historical moment in cinem...,[broken heart],"[got, witness, historical, moment, cinema, kil...","[got, witness, historical, moment, cinema, kil...","[0.2409971, 0.20214732, 0.16738108, 0.00604014...","[-0.001636048, 0.004795282, 0.00082498364, -0....","[0.2409971, 0.20214732, 0.16738108, 0.00604014...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",-10.289240
